# 0.1 RPC BAL RLP Estimation

This notebook estimates raw RLP BAL bytes using JSON-RPC `debug_traceBlockByNumber` with `prestateTracer`, following Toni's `eth-bal-analysis` builder logic.

It produces a BAL-only block summary. It does not pull calldata and does not join bandwidth components. The calldata + BAL + access-list + authorization + blob-hash join happens later in `0.3-bandwidth-content.ipynb`.

## BAL Encoding

Each BAL account entry is encoded as:

```text
[address, storage_writes, storage_reads, balance_changes, nonce_changes, code_changes]
```

The block-level BAL is built by set union across transaction traces, not by summing per-transaction BAL sizes.

In [1]:
import importlib
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import sim.rpc_bal as rpc_bal
rpc_bal = importlib.reload(rpc_bal)

BAL_SEMANTICS = rpc_bal.BAL_SEMANTICS
build_rpc_bal_for_block = rpc_bal.build_rpc_bal_for_block

load_dotenv(PROJECT_ROOT / ".env")

RPC_URL = os.environ.get(
    "ETHNODEOPS_RPC",
    "https://erigon.mainnet.rpc.ethnodeops.xyz",
)
ETHNODEOPS_API_KEY = os.environ.get("ETHNODEOPS_API_KEY") or os.environ.get("hoodi_api_key")
if not ETHNODEOPS_API_KEY:
    raise RuntimeError("Missing ETHNODEOPS_API_KEY in .env")

RPC_HEADERS = {"X-API-Key": ETHNODEOPS_API_KEY}
RPC_PROVIDER_LABEL = "ethnodeops_erigon_mainnet"
print("Loaded ETHNODEOPS_API_KEY; using ethnodeops Erigon mainnet RPC")

Loaded ETHNODEOPS_API_KEY; using ethnodeops Erigon mainnet RPC


In [2]:
START_BLOCK = 24_120_001
N_BLOCKS = 500
BLOCKS = list(range(START_BLOCK, START_BLOCK + N_BLOCKS))
SUMMARY_CSV = PROJECT_ROOT / "data" / f"rpc_bal_summary_{min(BLOCKS)}_{max(BLOCKS)}.csv"

INCLUDE_READS = True

# True estimates the fuller EIP-7928 block-level BAL payload.
INCLUDE_SYSTEM_CHANGES = True
BAL_SEMANTICS_VERSION = BAL_SEMANTICS

WRITE_CSV = True
WRITE_RLP = False

min(BLOCKS), max(BLOCKS), len(BLOCKS)

(24120001, 24120500, 500)

In [3]:
# BAL estimation is independent of calldata. The bandwidth join happens in 0.3.
pd.DataFrame({"block_number": [min(BLOCKS), max(BLOCKS)], "role": ["start", "end"]})

,block_number,role
0,24120001,start
1,24120500,end


In [4]:
summary_cols = [
    "block_number",
    "rpc_provider",
    "bal_semantics",
    "include_reads",
    "include_system_changes",
    "bal_rlp_bytes",
    "accounts",
    "storage_write_slots",
    "storage_write_changes",
    "storage_reads",
    "balance_changes",
    "nonce_changes",
    "code_changes",
    "code_bytes",
    "storage_writes_rlp_bytes",
    "storage_reads_rlp_bytes",
    "balance_changes_rlp_bytes",
    "nonce_changes_rlp_bytes",
    "code_changes_rlp_bytes",
    "account_shell_rlp_bytes",
]

rows = []
if SUMMARY_CSV.exists():
    prior = pd.read_csv(SUMMARY_CSV)
    required_cache_cols = {"rpc_provider", "bal_semantics", "include_reads", "include_system_changes"}
    if required_cache_cols.issubset(prior.columns):
        prior = prior[
            (prior["rpc_provider"] == RPC_PROVIDER_LABEL)
            & (prior["bal_semantics"] == BAL_SEMANTICS_VERSION)
            & (prior["include_reads"].astype(bool) == INCLUDE_READS)
            & (prior["include_system_changes"].astype(bool) == INCLUDE_SYSTEM_CHANGES)
        ]
    else:
        prior = prior.iloc[0:0]
    available_summary_cols = [col for col in summary_cols if col in prior.columns]
    rows.extend(prior[available_summary_cols].to_dict("records"))

seen = {int(row["block_number"]) for row in rows}
rlp_outputs = {}

for block_number in BLOCKS:
    if int(block_number) in seen:
        print(f"Skipping block {block_number}; already in {SUMMARY_CSV.name}")
        continue
    print(f"Building RPC BAL for block {block_number}...")
    result = build_rpc_bal_for_block(
        RPC_URL,
        block_number,
        rpc_headers=RPC_HEADERS,
        include_reads=INCLUDE_READS,
        include_system_changes=INCLUDE_SYSTEM_CHANGES,
    )
    row = result.summary.as_dict()
    row["rpc_provider"] = RPC_PROVIDER_LABEL
    rows.append({column: row.get(column, 0) for column in summary_cols})
    seen.add(int(block_number))
    rlp_outputs[block_number] = result.rlp_bytes
    if WRITE_CSV:
        data_dir = PROJECT_ROOT / "data"
        data_dir.mkdir(exist_ok=True)
        pd.DataFrame(rows).reindex(columns=summary_cols).drop_duplicates("block_number", keep="last").sort_values("block_number").to_csv(SUMMARY_CSV, index=False)

summary = pd.DataFrame(rows).reindex(columns=summary_cols).drop_duplicates("block_number", keep="last").sort_values("block_number")
display(summary)

if WRITE_CSV:
    data_dir = PROJECT_ROOT / "data"
    data_dir.mkdir(exist_ok=True)
    summary.to_csv(SUMMARY_CSV, index=False)
    print(SUMMARY_CSV)

if WRITE_RLP:
    suffix = "with_reads" if INCLUDE_READS else "without_reads"
    for block_number, payload in rlp_outputs.items():
        data_dir = PROJECT_ROOT / "data"
        data_dir.mkdir(exist_ok=True)
        out = data_dir / f"rpc_bal_{block_number}_{suffix}.rlp"
        out.write_bytes(payload)
        print(out)

Skipping block 24120001; already in rpc_bal_summary_24120001_24120500.csv
Skipping block 24120002; already in rpc_bal_summary_24120001_24120500.csv
Skipping block 24120003; already in rpc_bal_summary_24120001_24120500.csv
Skipping block 24120004; already in rpc_bal_summary_24120001_24120500.csv
Skipping block 24120005; already in rpc_bal_summary_24120001_24120500.csv
Skipping block 24120006; already in rpc_bal_summary_24120001_24120500.csv
Skipping block 24120007; already in rpc_bal_summary_24120001_24120500.csv
Skipping block 24120008; already in rpc_bal_summary_24120001_24120500.csv
Skipping block 24120009; already in rpc_bal_summary_24120001_24120500.csv
Skipping block 24120010; already in rpc_bal_summary_24120001_24120500.csv
Skipping block 24120011; already in rpc_bal_summary_24120001_24120500.csv
Skipping block 24120012; already in rpc_bal_summary_24120001_24120500.csv
Skipping block 24120013; already in rpc_bal_summary_24120001_24120500.csv
Skipping block 24120014; already in rp

,block_number,rpc_provider,bal_semantics,include_reads,include_system_changes,bal_rlp_bytes,accounts,storage_write_slots,storage_write_changes,storage_reads,balance_changes,nonce_changes,code_changes,code_bytes,storage_writes_rlp_bytes,storage_reads_rlp_bytes,balance_changes_rlp_bytes,nonce_changes_rlp_bytes,code_changes_rlp_bytes,account_shell_rlp_bytes
0,24120001,ethnodeops_erigon_mainnet,eip7928_pre_tx_post_indices_v1,True,True,339327,892,3145,3310,2009,962,350,4,1358,233907,66719,11942,2117,1392,23250
1,24120002,ethnodeops_erigon_mainnet,eip7928_pre_tx_post_indices_v1,True,True,72309,432,482,516,560,474,182,2,46,35712,18712,5521,1036,56,11272
2,24120003,ethnodeops_erigon_mainnet,eip7928_pre_tx_post_indices_v1,True,True,186956,1088,1133,1219,1608,1299,493,2,1217,85070,53560,15894,2975,1233,28224
3,24120004,ethnodeops_erigon_mainnet,eip7928_pre_tx_post_indices_v1,True,True,117839,647,763,793,995,655,236,2,3152,55731,33198,7534,1307,3166,16903
4,24120005,ethnodeops_erigon_mainnet,eip7928_pre_tx_post_indices_v1,True,True,81624,527,512,545,601,644,252,2,1110,37902,20062,7465,1400,1125,13670
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,24120496,ethnodeops_erigon_mainnet,eip7928_pre_tx_post_indices_v1,True,True,270342,963,2406,2536,1471,1159,430,3,69,179771,49032,13854,2540,83,25062
496,24120497,ethnodeops_erigon_mainnet,eip7928_pre_tx_post_indices_v1,True,True,331039,810,3053,3178,2035,1053,378,3,1085,226361,67511,12883,2169,1110,21005
497,24120498,ethnodeops_erigon_mainnet,eip7928_pre_tx_post_indices_v1,True,True,251143,857,2249,2365,1457,953,309,2,78,167126,48434,11375,1799,88,22321
498,24120499,ethnodeops_erigon_mainnet,eip7928_pre_tx_post_indices_v1,True,True,27179,221,176,181,154,267,99,1,23,12753,5162,2977,517,27,5743


/Users/william/PycharmProjects/eip-7999-research/data/rpc_bal_summary_24120001_24120500.csv


In [5]:
# Optional local calibration against nerolation/eth-bal-analysis raw RLP samples.
sample_dir = Path("/private/tmp/eth-bal-analysis/bal_raw/rlp")
calibration_rows = []
if sample_dir.exists():
    suffix = "with_reads" if INCLUDE_READS else "without_reads"
    for block_number in BLOCKS:
        sample = sample_dir / f"{block_number}_{suffix}.rlp"
        if sample.exists():
            sample_bytes = sample.stat().st_size
            row = summary[summary["block_number"] == block_number].iloc[0]
            calibration_rows.append({
                "block_number": block_number,
                "rpc_bal_rlp_bytes": int(row["bal_rlp_bytes"]),
                "sample_bal_rlp_bytes": sample_bytes,
                "delta_bytes": int(row["bal_rlp_bytes"]) - sample_bytes,
            })

calibration = pd.DataFrame(calibration_rows)
display(calibration)

""
